### 1. Libraries

In [12]:
# Data handling
import numpy as np
import pandas as pd

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

### 2. Data Loading

In [30]:
df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv("data/test.csv")

### 3. Feature Engineering and Model Training

In [ ]:
target_col = 'Churn'

#### 3.1. Logistic Regression

In [31]:
df_train_log = df_train.copy()
df_train_log[target_col] = df_train_log[target_col].map({"No": 0, "Yes": 1})

In [32]:
df_test_log = df_test.copy()

##### 3.1.1. Feature Engineering

**Household Type**

When customer have either `Partner` or `Dependent` or both, the churn rates significantly go down, possibly because of more sharing of the service, leading to higher number of users per household, and thus loyalty. 

In [33]:
def get_household_type(row):
    if row['Partner'] == 'No' and row['Dependents'] == 'No':
        return 'Single'
    elif row['Partner'] == 'Yes' and row['Dependents'] == 'No':
        return 'Couple'
    elif row['Partner'] == 'Yes' and row['Dependents'] == 'Yes':
        return 'Family'
    else:
        return 'Single Parent'

df_train_log['household_type'] = df_train_log.apply(get_household_type, axis=1)
df_test_log['household_type'] = df_test_log.apply(get_household_type, axis=1)

**Service Engagement Score**

The more protection services a customer have, the less the churn rate. 

In [34]:
services = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]

df_train_log["num_services"] = df_train_log[services].apply(lambda x: (x == "Yes").sum(), axis=1)
df_test_log["num_services"] = df_test_log[services].apply(lambda x: (x == "Yes").sum(), axis=1)

**High Churn Rate Signal Group**

In [35]:
# Create the specific high-risk flag
df_train_log['high_risk_digital_senior'] = (
    (df_train_log['SeniorCitizen'] == 1) & 
    (df_train_log['PaymentMethod'] == 'Electronic check') & 
    (df_train_log['PaperlessBilling'] == 'Yes')
).astype(int)

# Apply the same flag to the test set
df_test_log['high_risk_digital_senior'] = (
    (df_test_log['SeniorCitizen'] == 1) & 
    (df_test_log['PaymentMethod'] == 'Electronic check') & 
    (df_test_log['PaperlessBilling'] == 'Yes')
).astype(int)

##### 3.1.2. Model Training

In [37]:
X = df_train_log.drop(columns=[target_col, "id"]).copy()
y = df_train_log[target_col].copy()
X_test = df_test_log.drop(columns=["id"]).copy()

In [38]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

In [14]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]
)

preprocessor_unscaled = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]
)

In [15]:
def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

In [16]:
from copy import deepcopy

def run_oof(model, X, y, X_test, cv, model_name="model"):
    oof_pred = np.zeros(len(X))
    test_pred_folds = np.zeros((len(X_test), cv.get_n_splits()))
    fold_metrics = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]
        y_valid_fold = y.iloc[valid_idx]

        model_fold = deepcopy(model)
        model_fold.fit(X_train_fold, y_train_fold)

        valid_prob = model_fold.predict_proba(X_valid_fold)[:, 1]
        test_prob = model_fold.predict_proba(X_test)[:, 1]

        oof_pred[valid_idx] = valid_prob
        test_pred_folds[:, fold - 1] = test_prob

        fold_result = evaluate_predictions(y_valid_fold, valid_prob, threshold=0.5)
        fold_result["fold"] = fold
        fold_metrics.append(fold_result)

        print(
            f"{model_name} | Fold {fold} | "
            f"AUC={fold_result['roc_auc']:.4f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"Precision={fold_result['precision']:.4f} | "
            f"Recall={fold_result['recall']:.4f}"
        )

    fold_metrics_df = pd.DataFrame(fold_metrics)
    overall_metrics = evaluate_predictions(y, oof_pred, threshold=0.5)

    print(f"\n{model_name} OOF Overall:")
    for k, v in overall_metrics.items():
        print(f"{k}: {v:.4f}")

    return {
        "oof_pred": oof_pred,
        "test_pred": test_pred_folds.mean(axis=1),
        "fold_metrics": fold_metrics_df,
        "overall_metrics": overall_metrics
    }

In [17]:
def find_best_threshold(y_true, y_prob, metric="f1", thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.01)

    rows = []
    for t in thresholds:
        scores = evaluate_predictions(y_true, y_prob, threshold=t)
        scores["threshold"] = t
        rows.append(scores)

    result_df = pd.DataFrame(rows)
    best_row = result_df.sort_values(metric, ascending=False).iloc[0]
    return result_df, best_row

In [18]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [19]:
from sklearn.linear_model import LogisticRegression

In [20]:
logreg_candidates = {
    "logreg_l2_default": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=2000,
            random_state=42
        ))
    ]),
    
    "logreg_l2_balanced": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ]),
    
    "logreg_l1_balanced": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(
            penalty="l1",
            C=0.5,
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ]),
    
    "logreg_l2_stronger_reg": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(
            penalty="l2",
            C=0.3,
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ])
}

In [39]:
logreg_results = {}

for name, model in logreg_candidates.items():
    print(f"\n{'='*70}\nRunning {name}\n{'='*70}")
    result = run_oof(model, X, y, X_test, cv, model_name=name)
    logreg_results[name] = result


Running logreg_l2_default
logreg_l2_default | Fold 1 | AUC=0.9075 | F1=0.6668 | Precision=0.6827 | Recall=0.6515
logreg_l2_default | Fold 2 | AUC=0.9089 | F1=0.6731 | Precision=0.6852 | Recall=0.6614
logreg_l2_default | Fold 3 | AUC=0.9081 | F1=0.6708 | Precision=0.6846 | Recall=0.6576
logreg_l2_default | Fold 4 | AUC=0.9091 | F1=0.6705 | Precision=0.6904 | Recall=0.6517
logreg_l2_default | Fold 5 | AUC=0.9061 | F1=0.6691 | Precision=0.6839 | Recall=0.6550

logreg_l2_default OOF Overall:
accuracy: 0.8546
precision: 0.6854
recall: 0.6555
f1: 0.6701
roc_auc: 0.9079

Running logreg_l2_balanced
logreg_l2_balanced | Fold 1 | AUC=0.9074 | F1=0.6737 | Precision=0.5458 | Recall=0.8800
logreg_l2_balanced | Fold 2 | AUC=0.9088 | F1=0.6715 | Precision=0.5418 | Recall=0.8829
logreg_l2_balanced | Fold 3 | AUC=0.9080 | F1=0.6720 | Precision=0.5438 | Recall=0.8792
logreg_l2_balanced | Fold 4 | AUC=0.9090 | F1=0.6748 | Precision=0.5476 | Recall=0.8791
logreg_l2_balanced | Fold 5 | AUC=0.9059 | F1=0.6

In [40]:
logreg_summary = pd.DataFrame({
    name: result["overall_metrics"]
    for name, result in logreg_results.items()
}).T.sort_values("roc_auc", ascending=False)

logreg_summary

,accuracy,precision,recall,f1,roc_auc
logreg_l2_default,0.854638,0.685360,0.655455,0.670074,0.907938
logreg_l2_balanced,0.806895,0.544102,0.879313,0.672237,0.907794
logreg_l1_balanced,0.806895,0.544102,0.879320,0.672239,0.907793
logreg_l2_stronger_reg,0.806895,0.544102,0.879313,0.672237,0.907791


In [41]:
best_logreg_name = logreg_summary.index[0]
best_logreg_oof = logreg_results[best_logreg_name]["oof_pred"]
best_logreg_test = logreg_results[best_logreg_name]["test_pred"]

In [42]:
threshold_df, best_threshold_row = find_best_threshold(y, best_logreg_oof, metric="f1")
best_threshold_row

accuracy     0.839212
precision    0.609923
recall       0.793584
f1           0.689737
roc_auc      0.907938
threshold    0.350000
Name: 30, dtype: float64

#### 3.2. XGBoost

In [43]:
from xgboost import XGBClassifier

In [44]:
xgb_base = Pipeline([
    ("preprocessor", preprocessor_unscaled),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_weight=1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

In [45]:
xgb_result = run_oof(xgb_base, X, y, X_test, cv, model_name="xgb_base")

xgb_base | Fold 1 | AUC=0.9148 | F1=0.6714 | Precision=0.7088 | Recall=0.6377
xgb_base | Fold 2 | AUC=0.9157 | F1=0.6742 | Precision=0.7077 | Recall=0.6437
xgb_base | Fold 3 | AUC=0.9151 | F1=0.6740 | Precision=0.7118 | Recall=0.6400
xgb_base | Fold 4 | AUC=0.9162 | F1=0.6740 | Precision=0.7166 | Recall=0.6362
xgb_base | Fold 5 | AUC=0.9134 | F1=0.6710 | Precision=0.7092 | Recall=0.6367

xgb_base OOF Overall:
accuracy: 0.8601
precision: 0.7108
recall: 0.6389
f1: 0.6729
roc_auc: 0.9150


In [46]:
xgb_candidates = {
    "xgb_depth4": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("model", XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ]),
    
    "xgb_depth5": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("model", XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ]),
    
    "xgb_stronger_reg": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("model", XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_alpha=0.2,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

In [47]:
xgb_results = {}

for name, model in xgb_candidates.items():
    print(f"\n{'='*70}\nRunning {name}\n{'='*70}")
    result = run_oof(model, X, y, X_test, cv, model_name=name)
    xgb_results[name] = result


Running xgb_depth4
xgb_depth4 | Fold 1 | AUC=0.9153 | F1=0.6734 | Precision=0.7103 | Recall=0.6401
xgb_depth4 | Fold 2 | AUC=0.9162 | F1=0.6758 | Precision=0.7096 | Recall=0.6450
xgb_depth4 | Fold 3 | AUC=0.9156 | F1=0.6748 | Precision=0.7127 | Recall=0.6407
xgb_depth4 | Fold 4 | AUC=0.9167 | F1=0.6753 | Precision=0.7165 | Recall=0.6385
xgb_depth4 | Fold 5 | AUC=0.9139 | F1=0.6727 | Precision=0.7101 | Recall=0.6391

xgb_depth4 OOF Overall:
accuracy: 0.8607
precision: 0.7118
recall: 0.6407
f1: 0.6744
roc_auc: 0.9155

Running xgb_depth5
xgb_depth5 | Fold 1 | AUC=0.9158 | F1=0.6757 | Precision=0.7125 | Recall=0.6425
xgb_depth5 | Fold 2 | AUC=0.9167 | F1=0.6767 | Precision=0.7113 | Recall=0.6454
xgb_depth5 | Fold 3 | AUC=0.9161 | F1=0.6778 | Precision=0.7145 | Recall=0.6447
xgb_depth5 | Fold 4 | AUC=0.9171 | F1=0.6759 | Precision=0.7169 | Recall=0.6394
xgb_depth5 | Fold 5 | AUC=0.9144 | F1=0.6741 | Precision=0.7108 | Recall=0.6410

xgb_depth5 OOF Overall:
accuracy: 0.8613
precision: 0.713

In [48]:
xgb_summary = pd.DataFrame({
    name: result["overall_metrics"]
    for name, result in xgb_results.items()
}).T.sort_values("roc_auc", ascending=False)

xgb_summary

,accuracy,precision,recall,f1,roc_auc
xgb_depth5,0.861311,0.713183,0.642609,0.676059,0.916006
xgb_depth4,0.860670,0.711826,0.640711,0.674399,0.915531
xgb_stronger_reg,0.860123,0.710478,0.639493,0.673119,0.915039


In [49]:
best_xgb_name = xgb_summary.index[0]
best_xgb_oof = xgb_results[best_xgb_name]["oof_pred"]
best_xgb_test = xgb_results[best_xgb_name]["test_pred"]

#### 3.3. Neural Network MLP